# 特徴量エンジニアリング: 採用した13個の特徴量の考え方とコード

採用した特徴量を、READMEの「2. LightGBMでベースライン作成と特徴量エンジニアリング」と同じ3つの作り方に分けて、考えた順に並べる。

- 1つの特徴量ごとに **【仮説】→【確認】(その場で集計)→【結果】→【施策】(特徴量を作るコード)** の順に書く。改善幅は採否の判定(LightGBM、READMEの3の節の基準)で測った値。
- ここのコードは説明のために書き下したもの。学習に使うコードは `feature_definitions.py` にあり、**最後のセルで両方が全行で同じ特徴量を作ることを照合する**(食い違えばエラーで止まる)。
- 試したが採用しなかった約90個の候補は `rejected_feature_candidates.py` にある。
- 実行はリポジトリのルートで行う(`data/train.csv`・`data/test.csv`と、`make_expected_rating.py`で作る`data/aux_ev_*.npy`を使う)。

In [1]:
import os
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

if not os.path.exists("data/train.csv"):  # ノートブックのフォルダで開いたときはリポジトリのルートに移る
    os.chdir("../..")
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = train["satisfaction"].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")


def satisfaction_rate(by):
    """グループごとの満足率と人数の表(確認用)。"""
    return train.assign(_y=y).groupby(by, observed=True)["_y"].agg(満足率="mean", 人数="size").round(3)

train 699,635行・test 299,844行、満足率 0.444


## ① 条件で絞る(ゲート特徴量)

点数が効く客層だけに値を持たせ、ほかの客は空欄(欠損)にする。木のモデルは「空欄かどうか」でも分岐できるので、「この客層ではこの点数が効く」という関係を1列で渡せる。

### 1. 出張客の快適さ(`comfort_gate_business`)

**【仮説】** 私用客は、座席や機内エンタメが快適でも満足しないのでは?(EDA 5節: 私用旅行はクラスに関係なく満足率9.5〜11.2%)

**【確認】** 快適さ(座席・足元の広さ・機内エンタメの点数の平均。0点は「評価なし」の可能性があるので除く)を四捨五入し、旅行目的ごとの満足率を見る。

In [2]:
COMFORT_RATINGS = ["Seat comfort", "Leg room service", "Inflight entertainment"]


def comfort(df):
    return df[COMFORT_RATINGS].replace(0, np.nan).mean(axis=1)


satisfaction_rate(["Type of Travel", comfort(train).round()])["満足率"].unstack()

,1.0,2.0,3.0,4.0,5.0
Type of Travel,,,,,
Business travel,0.125,0.147,0.324,0.750,0.887
Personal Travel,0.099,0.093,0.091,0.103,0.119


**【結果】** 出張客は、快適さが4点で満足率75.0%、5点で88.7%と大きく上がる(3点以下は12.5〜32.4%)。私用客は、どの点数でも9.1〜11.9%でほぼ横ばい。快適さは出張客にだけ効く。

**【施策】** 快適さを出張客だけに持たせ、私用客は空欄にする。改善幅 **+0.000063**(3seed平均)。ゲートなしの快適さ(全員に値を持たせる版)は効かなかったので、効果はゲートによるもの。

In [3]:
def comfort_gate_business(df):
    return pd.DataFrame({"comfort_business": comfort(df).where(df["Type of Travel"] == "Business travel")})

### 2. 常連かつ出張客の快適さ(`comfort_gate_loyal_business`)

**【仮説】** 出張客の中でも、常連でない客(初めて・たまにしか乗らない客)には快適さが効かないのでは?(EDA 8節)

**【確認】** 出張客に絞り、常連かどうか×快適さの満足率を見る。

In [4]:
business = train["Type of Travel"] == "Business travel"
train[business].assign(_y=y[business], comfort=comfort(train[business]).round()).pivot_table(
    index="Customer Type", columns="comfort", values="_y", aggfunc="mean").round(3)

comfort,1.0,2.0,3.0,4.0,5.0
Customer Type,,,,,
Loyal Customer,0.081,0.116,0.388,0.873,0.956
disloyal Customer,0.155,0.187,0.205,0.209,0.228


**【結果】** 常連の出張客は、快適さが4点で87.3%、5点で95.6%まで上がる。常連でない出張客は、1点15.5%から5点22.8%までしか上がらない。快適さが効くのは常連の出張客に限られる。

**【施策】** 快適さを常連の出張客だけに持たせる。改善幅 **+0.000070**(3seed平均)。SHAP(予測への寄与)で全特徴量の1位になった。

In [5]:
def comfort_gate_loyal_business(df):
    mask = (df["Type of Travel"] == "Business travel") & (df["Customer Type"] == "Loyal Customer")
    return pd.DataFrame({"comfort_loyal_business": comfort(df).where(mask)})

### 3. 出張客の機内サービス(`gate_onboard`)

**【仮説】** 快適さ以外の評価項目でも、出張客にだけ効くものがあるのでは?

**【確認】** 機内サービスの点数だけで満足/不満を並べ分けたときのAUC(0.5=当てずっぽう、1=完璧)を、出張客・私用客で比べる。

In [6]:
personal = ~business
pd.Series({"出張客": roc_auc_score(y[business], train.loc[business, "On-board service"]),
           "私用客": roc_auc_score(y[personal], train.loc[personal, "On-board service"])}, name="機内サービスの単変量AUC").round(3)

出張客    0.759
私用客    0.509
Name: 機内サービスの単変量AUC, dtype: float64

**【結果】** 単変量AUCは、出張客0.759、私用客0.509(ほぼ当てずっぽう)。機内サービスの点数も、出張客にだけ効く。

**【施策】** 機内サービスの点数を出張客だけに持たせる。改善幅 **+0.000045**(3seed平均)。

In [7]:
def gate_onboard(df):
    return pd.DataFrame({"On-board service_business": df["On-board service"].where(df["Type of Travel"] == "Business travel")})

## ② 似た乗客で数える・平均する

飛行距離は値の種類が多く(3,474種類)、同じ距離=同じ路線と考えられる。同じ路線・同じ客層の乗客をまとめて、満足率・人数・評価の平均などを特徴量にする。

満足率(Target Encoding)は答えを使うので、**自分の答えが混ざらないように**作る(下の`target_encode_in_folds`)。人数が少ない値は、全体の満足率に寄せて極端な値を防ぐ(強さm=3)。

In [8]:
def target_encode(key_fit, y_fit, key_apply, m=3):
    """key_fitの値ごとの満足率を、人数が少ないほど全体の満足率に寄せて計算し、key_applyに当てはめる。"""
    prior = y_fit.mean()
    stats = pd.DataFrame({"key": key_fit, "y": y_fit}).groupby("key")["y"].agg(["sum", "count"])
    return pd.Series(key_apply).map((stats["sum"] + m * prior) / (stats["count"] + m)).fillna(prior).values


def target_encode_in_folds(key_fit, y_fit, key_others, seed, m=3):
    """学習用の行は、その中でさらに5分割し、自分を含まない4つで計算した満足率を入れる。
    検証用・testの行(key_others)には、学習用の行全体の満足率を入れる。"""
    encoded = np.zeros(len(key_fit))
    for inner_fit, inner_apply in StratifiedKFold(5, shuffle=True, random_state=seed).split(key_fit, y_fit):
        encoded[inner_apply] = target_encode(key_fit[inner_fit], y_fit[inner_fit], key_fit[inner_apply], m)
    return encoded, [target_encode(key_fit, y_fit, key, m) for key in key_others]


def oof_target_encoding_auc(key):
    """確認用: 5分割の外側で満足率を当てはめ、その満足率だけで満足/不満を並べ分けたときのAUC。"""
    oof = np.zeros(len(key))
    for fit, apply in StratifiedKFold(5, shuffle=True, random_state=42).split(key, y):
        oof[apply] = target_encode(key[fit], y[fit], key[apply])
    return roc_auc_score(y, oof)

### 4. 飛行距離の値ごとの満足率(`te_distance_m3`)

**【仮説】** 飛行距離は「長いほど満足」というまっすぐな関係ではなく、値ごと(路線ごと)に満足率のクセがあるのでは?

**【確認】** 飛行距離の値の種類の数と、「距離そのもの」と「距離の値ごとの満足率」それぞれの単変量AUCを比べる。

In [9]:
distance = train["Flight Distance"].values
pd.Series({"値の種類": train["Flight Distance"].nunique(),
           "距離そのもののAUC": round(roc_auc_score(y, distance), 3),
           "値ごとの満足率のAUC": round(oof_target_encoding_auc(distance), 3)})

値の種類           3474.000
距離そのもののAUC        0.696
値ごとの満足率のAUC       0.743
dtype: float64

**【結果】** 飛行距離の値は3,474種類。距離そのもののAUC 0.696に対し、値ごとの満足率は0.743と高い。距離の大小より、値(路線)ごとのクセの方が満足度をよく表す。

**【施策】** 特徴量の作成では距離の値(キー)だけを置き、学習のfoldごとに`target_encode_in_folds`で満足率に置き換える。改善幅 **+0.00136**(1seed。特徴量エンジニアリングで最大の改善、Public LBでも+0.00122)。

In [10]:
def te_distance_m3(df):
    return pd.DataFrame({"te_distance_m3": df["Flight Distance"]})  # 満足率への置き換えは学習ループの中で行う

### 5. 飛行距離×座席クラスの満足率(`te_dist_class`)

**【仮説】** 同じ路線でも、ビジネス席とエコノミー席では満足率が違うのでは?(EDA 5節: 距離の効き方は客層で違う)

**【確認】** 「距離だけ」と「距離×座席クラス」の満足率の単変量AUCを比べる。

In [11]:
distance_class = (train["Flight Distance"].astype(str) + "|" + train["Class"]).values
pd.Series({"距離だけの満足率のAUC": round(oof_target_encoding_auc(distance), 3),
           "距離×座席クラスの満足率のAUC": round(oof_target_encoding_auc(distance_class), 3)})

距離だけの満足率のAUC        0.743
距離×座席クラスの満足率のAUC    0.839
dtype: float64

**【結果】** 距離だけの満足率のAUC 0.743に対し、距離×座席クラスでは0.839まで上がる。同じ路線でも座席クラスで満足率が大きく違う。

**【施策】** 距離と座席クラスをつないだキー(例: `"1235|Business"`)を置き、学習ループの中で満足率に置き換える。改善幅 **+0.000060**(3seed平均)。

In [12]:
def te_dist_class(df):
    key = df[["Flight Distance", "Class"]].fillna(-1).astype(str).agg("|".join, axis=1)
    return pd.DataFrame({"te_dist_class": key})

### 6. 飛行距離×旅行目的の出現回数(`freq_dist_travel`)

**【仮説】** 出張客の多い路線(ビジネス路線)かどうかは、満足率と関係するのでは?(2つ前のコンペで、値の出現回数を足すと効いた経験から。候補は13個試した)

**【確認】** 同じ距離・同じ旅行目的の乗客がtrain+testに何人いるかを数え、人数を5つの区間に分けて満足率を見る。

In [13]:
def count_in_train_and_test(df, cols):
    keys = [pd.read_csv(f, usecols=cols)[cols].fillna(-1).astype(str).agg("|".join, axis=1) for f in ("data/train.csv", "data/test.csv")]
    counts = pd.concat(keys).value_counts()
    return df[cols].fillna(-1).astype(str).agg("|".join, axis=1).map(counts).values


dist_travel_count = count_in_train_and_test(train, ["Flight Distance", "Type of Travel"])
satisfaction_rate(["Type of Travel", pd.qcut(dist_travel_count, 5, duplicates="drop")])["満足率"].unstack()

,"(0.999, 190.0]","(190.0, 537.0]","(537.0, 1202.0]","(1202.0, 2340.0]","(2340.0, 7170.0]"
Type of Travel,,,,,
Business travel,0.705,0.613,0.542,0.545,0.503
Personal Travel,0.112,0.099,0.101,0.089,0.089


**【結果】** 出張客は、同じ路線・同じ目的の乗客が少ない(190人以下)ほど満足率が高く(70.5%)、多い(2,340人超)と50.3%まで下がる。私用客は8.9〜11.2%でほぼ横ばい。出張客の多い大きな路線ほど満足しにくい。

**【施策】** 人数をそのまま特徴量にする(答えは使わないので、train+testで数えてよい)。改善幅 **+0.000057**(3seed平均)。試した13個のうち、効いたのはこれだけ。

In [14]:
def freq_dist_travel(df):
    return pd.DataFrame({"freq_dist_travel": count_in_train_and_test(df, ["Flight Distance", "Type of Travel"])}, index=df.index)

In [15]:
def route_stat(df, col, stat):
    """同じ飛行距離(=同じ路線とみなす)の乗客の、colの平均や標準偏差(train+testで計算。答えは使わない)。"""
    both = pd.concat([pd.read_csv(f, usecols=["Flight Distance", col]) for f in ("data/train.csv", "data/test.csv")])
    return df["Flight Distance"].map(both.groupby("Flight Distance")[col].agg(stat)).values

### 7. 足元の広さの路線平均(`rp_legroom`)

**【仮説】** 路線ごとに機材(座席の間隔)が違い、足元の評判がその路線の満足度を表すのでは?(Discussion: 距離を路線IDとみなす)

**【確認】** 路線ごとの足元の点数の平均を5つの区間に分け、満足率を見る。

In [16]:
satisfaction_rate(pd.qcut(route_stat(train, "Leg room service", "mean"), 5))

,満足率,人数
"(-0.001, 3.238]",0.304,139985
"(3.238, 3.304]",0.284,140212
"(3.304, 3.407]",0.316,140509
"(3.407, 3.759]",0.515,140212
"(3.759, 5.0]",0.802,138717


**【結果】** 足元の路線平均が上位20%(3.76点超)の路線は満足率80.2%。下位60%の路線は28.4〜31.6%。路線ごとの足元の評判で満足率が大きく変わる。

**【施策】** 路線平均をそのまま特徴量にする。改善幅 **+0.000061**(3seed平均)。

In [17]:
def rp_legroom(df):
    return pd.DataFrame({"rp_legroom": route_stat(df, "Leg room service", "mean")}, index=df.index)

### 8. Wi-Fiの路線標準偏差(`rsd_wifi`)

**【仮説】** Wi-Fiの評価が0点(使っていない)と5点に割れる路線は、客層が混ざった路線なのでは?(公開ノートブックの「路線プロファイル」から)

**【確認】** 路線ごとのWi-Fiの点数の標準偏差(ばらつき)を5つの区間に分け、満足率を見る。

In [18]:
satisfaction_rate(pd.qcut(route_stat(train, "Inflight wifi service", "std"), 5))

,満足率,人数
"(-0.001, 1.164]",0.331,140131
"(1.164, 1.211]",0.326,140158
"(1.211, 1.258]",0.387,139848
"(1.258, 1.332]",0.493,141689
"(1.332, 2.887]",0.684,137723


**【結果】** Wi-Fiのばらつきが大きい路線(上位20%)は満足率68.4%、小さい路線(下位40%)は32.6〜33.1%。評価が割れる路線ほど満足率が高い。

**【施策】** 路線の標準偏差をそのまま特徴量にする。改善幅 **+0.000048**(3seed平均)。

In [19]:
def rsd_wifi(df):
    return pd.DataFrame({"rsd_wifi": route_stat(df, "Inflight wifi service", "std")}, index=df.index)

### 9. 客層グループ内の偏差(`ctx_ctc`、4個)

**【仮説】** 同じ遅延10分でも、遅延の少ない客層と多い客層では意味が違う。同じ客層の中で「平均より上か下か」が効くのでは?(公開ノートブックの48個の特徴量を、偏差だけの4個に絞った)

**【確認】** 座席クラス×旅行目的×常連かどうかの12グループで、到着遅延がグループ平均より多い人・少ない人の満足率を比べる。

In [20]:
GROUP_KEYS = ["Class", "Type of Travel", "Customer Type"]
GROUP_COLS = {"age": "Age", "dist": "Flight Distance", "dep": "Departure Delay in Minutes", "arr": "Arrival Delay in Minutes"}


def deviation_from_group_mean(df, col):
    both = pd.concat([pd.read_csv(f, usecols=GROUP_KEYS + [col]) for f in ("data/train.csv", "data/test.csv")])
    group_mean = both.groupby(GROUP_KEYS)[col].mean().rename("group_mean")
    return df[col].values - df[GROUP_KEYS].merge(group_mean, left_on=GROUP_KEYS, right_index=True, how="left")["group_mean"].values


arr_dev = deviation_from_group_mean(train, "Arrival Delay in Minutes")
satisfaction_rate([train["Class"], np.where(arr_dev > 0, "グループ平均より遅れた", "グループ平均以下")])["満足率"].unstack()

,グループ平均より遅れた,グループ平均以下
Class,,
Business,0.617,0.734
Eco,0.107,0.173
Eco Plus,0.158,0.251


**【結果】** どの座席クラスでも、グループ平均より遅れた人の満足率の方が低い(Business 73.4%→61.7%、Eco 17.3%→10.7%、Eco Plus 25.1%→15.8%)。同じ客層の中での相対的な遅れが、満足度と関係している。

**【施策】** 年齢・飛行距離・出発遅延・到着遅延の4つについて、グループ平均との差を特徴量にする。改善幅 **+0.0000375**(3seed平均)。

In [21]:
def ctx_ctc(df):
    return pd.DataFrame({f"ctx_ctc_{short}": deviation_from_group_mean(df, col) for short, col in GROUP_COLS.items()}, index=df.index)

## ③ モデルで予想する

### 10. サービス評価の期待値(`ev_boarding`・`ev_legroom`・`ev_checkin`・`ev_time`)

**【仮説】** 同じ「4点」でも、ほかの項目にも高い点を付けている人の4点と、ほかは低い人の4点では意味が違うのでは?「ほかの項目への答え方から見て、この人ならこの項目に何点を付けそうか」(期待値)を特徴量にすれば、その違いを渡せる(公開ノートブックのアイデア)。

**【施策】** 他の項目への答え方から、その項目の点数をLightGBMで予想する(`make_expected_rating.py`。5分割の外側で予想するので、自分の答えは使わない)。学習に時間がかかるので、前もって作った`data/aux_ev_*.npy`を読み込む。

**【確認】** オンライン搭乗が4点の人だけに絞り、期待値の高さで5つに分けて満足率を見る。

In [22]:
n_train = len(train)


def expected_rating(df, short_name):
    expected = np.load(f"data/aux_ev_{short_name}.npy")  # train+testの順に保存されている
    return expected[:n_train] if len(df) == n_train else expected[n_train:]


score4 = (train["Online boarding"] == 4).values
expected_boarding = expected_rating(train, "boarding")
pd.DataFrame({"期待値": expected_boarding[score4], "_y": y[score4]}).groupby(
    pd.qcut(expected_boarding[score4], 5), observed=True)["_y"].agg(満足率="mean", 人数="size").round(3)

,満足率,人数
"(-0.001, 3.953]",0.273,42190
"(3.953, 4.002]",0.433,42190
"(4.002, 4.341]",0.741,42190
"(4.341, 4.454]",0.825,42190
"(4.454, 4.983]",0.936,42190


**【結果】** 同じ4点でも、期待値が低い人(ほかの項目の点数も低い人)は満足率27.3%、期待値が高い人は93.6%。点数そのものが同じでも、期待値によって満足率が大きく違う。

サービス評価13項目を1つずつ判定し、4項目(オンライン搭乗・足元の広さ・チェックイン・時刻の便利さ)を採用した。改善幅は各 **+0.000037〜+0.000088**。

In [23]:
def expected_rating_features(df):
    return pd.DataFrame({f"ev_{name}": expected_rating(df, name) for name in ("boarding", "legroom", "checkin", "time")}, index=df.index)

## 学習に使うコードとの照合

ここで書いた関数と、学習に使う`feature_definitions.py`の`FEATURE_BUILDERS`が、train・testの全行で同じ特徴量を作るかを確かめる。満足率の特徴量(4・5)は、キーと、foldごとに満足率へ置き換えた値の両方を比べる。

In [24]:
sys.path.insert(0, "src/2_lightgbm_baseline_and_feature_engineering")
import feature_definitions as features  # noqa: E402

NOTEBOOK_BUILDERS = {
    "comfort_gate_business": comfort_gate_business, "comfort_gate_loyal_business": comfort_gate_loyal_business,
    "gate_onboard": gate_onboard, "te_distance_m3": te_distance_m3, "te_dist_class": te_dist_class,
    "freq_dist_travel": freq_dist_travel, "rp_legroom": rp_legroom, "rsd_wifi": rsd_wifi, "ctx_ctc": ctx_ctc,
}
assert set(NOTEBOOK_BUILDERS) | {"ev_boarding", "ev_legroom", "ev_checkin", "ev_time"} == set(features.ADOPTED_FEATURES)
for df_name, df in (("train", train), ("test", test)):
    for name, build in NOTEBOOK_BUILDERS.items():
        pd.testing.assert_frame_equal(build(df), features.FEATURE_BUILDERS[name](df), check_names=False)
    ev_module = pd.concat([features.FEATURE_BUILDERS[f"ev_{n}"](df) for n in ("boarding", "legroom", "checkin", "time")], axis=1)
    pd.testing.assert_frame_equal(expected_rating_features(df), ev_module)
    print(f"{df_name}: 13個とも一致")

# 満足率への置き換え(fold0で比べる)
fit_rows, valid_rows = next(StratifiedKFold(5, shuffle=True, random_state=42).split(train, y))
keys = pd.concat([te_distance_m3(train), te_dist_class(train)], axis=1)
module_fit, (module_valid,) = features.target_encode_in_folds(keys.iloc[fit_rows], y[fit_rows], [keys.iloc[valid_rows]], seed=0)
for name in keys.columns:
    nb_fit, (nb_valid,) = target_encode_in_folds(keys[name].values[fit_rows], y[fit_rows], [keys[name].values[valid_rows]], seed=0)
    assert np.array_equal(nb_fit, module_fit[name].values) and np.array_equal(nb_valid, module_valid[name].values), name
print("満足率への置き換えも一致")

train: 13個とも一致


test: 13個とも一致


満足率への置き換えも一致
